# 04 — Sentinel‑2 chipping (8‑day windows) + chip indices

Run the STAC chipping pipeline and compute per-chip indices into chip_indices.csv.


In [3]:
from __future__ import annotations

import os, sys, subprocess, json, re
from pathlib import Path
from datetime import datetime

REPO_ROOT = Path.cwd().parent
print("REPO_ROOT:", REPO_ROOT)

def sh(cmd: str, check: bool=True) -> None:
    """Run a shell command (prints it first)."""
    print("\n▶", cmd)
    subprocess.run(cmd, shell=True, check=check)

def pick_first_existing(*cands: str, root: Path = REPO_ROOT) -> Path:
    for c in cands:
        p = root / c
        if p.exists():
            return p
    return root / cands[0]

AOI_PATH = pick_first_existing("deployment/aoi/aoi.geojson", "deployment/aoi/oman_query1.geojson")

def require_exists(p: Path, what: str="path") -> Path:
    if not p.exists():
        raise FileNotFoundError(f"Missing {what}: {p}")
    return p

def newest_path(glob_pat: str) -> Path | None:
    paths = list(REPO_ROOT.glob(glob_pat))
    if not paths:
        return None
    paths.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return paths[0]

def show_tree(root: Path, max_lines: int=200) -> None:
    i = 0
    for p in sorted(root.rglob("*")):
        if i >= max_lines:
            print("... (truncated)")
            return
        if p.is_dir():
            continue
        rel = p.relative_to(root)
        print(rel)
        i += 1

AOI_PATH = pick_first_existing("deployment/aoi/aoi.geojson", "deployment/aoi/oman_query1.geojson")

START = "2025-03-01"
END   = "2025-03-31"

OUT_ROOT = REPO_ROOT / "deployment/outputs/by_plant/osm_way_386838289"
MONTH_TAG = "2025-03"
CHIPS_DIR = OUT_ROOT / MONTH_TAG / "chips"
CHIPS_DIR.mkdir(parents=True, exist_ok=True)

CLOUD = 30
PER_WINDOW = 1
SIZE = 640
STRIDE = 256

print("AOI:", AOI_PATH)
print("CHIPS_DIR:", CHIPS_DIR)


REPO_ROOT: /Users/ameerfiras/REDNET-ML
AOI: /Users/ameerfiras/REDNET-ML/deployment/aoi/aoi.geojson
CHIPS_DIR: /Users/ameerfiras/REDNET-ML/deployment/outputs/by_plant/osm_way_386838289/2025-03/chips


## 4.1 Chip S2 imagery


In [5]:
script = REPO_ROOT / "scripts/download/s2_chip_8day.py"

sh(
  f'python "{script}" '
  f'--aoi "{AOI_PATH}" --start "{START}" --end "{END}" '
  f'--cloud {CLOUD} --per_window {PER_WINDOW} '
  f'--size {SIZE} --stride {STRIDE} '
  f'--out "{CHIPS_DIR}"'
)


▶ python "/Users/ameerfiras/REDNET-ML/scripts/download/s2_chip_8day.py" --aoi "/Users/ameerfiras/REDNET-ML/deployment/aoi/aoi.geojson" --start "2025-03-01" --end "2025-03-31" --cloud 30 --per_window 1 --size 640 --stride 256 --out "/Users/ameerfiras/REDNET-ML/deployment/outputs/by_plant/osm_way_386838289/2025-03/chips"


/Users/ameerfiras/miniforge3/envs/rednet-ml/lib/python3.11/site-packages/pystac_client/client.py:181: FutureWarning: The `ignore_conformance` option is deprecated and will be removed in the next major release. Instead use `set_conforms_to` or `add_conforms_to` to control behavior.
  warnings.warn(


Found 4 items across 8-day windows.
✅ S2B_MSIL2A_20250308T064629_R020_T40QEM_20250308T085847: wrote 0 tiles → /Users/ameerfiras/REDNET-ML/deployment/outputs/by_plant/osm_way_386838289/2025-03/chips/tiles_png
✅ S2C_MSIL2A_20250316T065651_R063_T39QZD_20250316T122619: wrote 0 tiles → /Users/ameerfiras/REDNET-ML/deployment/outputs/by_plant/osm_way_386838289/2025-03/chips/tiles_png
✅ S2C_MSIL2A_20250323T064651_R020_T40QDH_20250323T123317: wrote 0 tiles → /Users/ameerfiras/REDNET-ML/deployment/outputs/by_plant/osm_way_386838289/2025-03/chips/tiles_png
✅ S2B_MSIL2A_20250331T065619_R063_T40QBK_20250331T105206: wrote 0 tiles → /Users/ameerfiras/REDNET-ML/deployment/outputs/by_plant/osm_way_386838289/2025-03/chips/tiles_png

Done. Total tiles: 0 → /Users/ameerfiras/REDNET-ML/deployment/outputs/by_plant/osm_way_386838289/2025-03/chips/index.csv


## 4.2 Compute chip indices


In [8]:
script = REPO_ROOT / "scripts/download/s2_compute_chip_indices.py"

sh(f'python "{script}" --folder "{CHIPS_DIR}"')

import pandas as pd
csv = CHIPS_DIR / "chip_indices.csv"
print("chip_indices.csv exists:", csv.exists())
if csv.exists():
    df = pd.read_csv(csv)
    display(df.head(3))
    print("rows:", len(df), "cols:", len(df.columns))



▶ python "/Users/ameerfiras/REDNET-ML/scripts/download/s2_compute_chip_indices.py" --folder "/Users/ameerfiras/REDNET-ML/deployment/outputs/by_plant/osm_way_386838289/2025-03/chips"


/Users/ameerfiras/miniforge3/envs/rednet-ml/lib/python3.11/site-packages/pystac_client/client.py:181: FutureWarning: The `ignore_conformance` option is deprecated and will be removed in the next major release. Instead use `set_conforms_to` or `add_conforms_to` to control behavior.
  warnings.warn(


✓ Wrote /Users/ameerfiras/REDNET-ML/deployment/outputs/by_plant/osm_way_386838289/2025-03/chips/chip_indices.csv
chip_indices.csv exists: True


,tile,scene_id,datetime,ndwi_mean,ndwi_std,fai_mean,fai_std,rednir_mean,rednir_std,valid_px


rows: 0 cols: 10
